# Eval 2 — Vídeos DQN / PPO / Best Agent

Genera un vídeo de conducción para cada modelo entrenado, cargándolos desde `models/`.

| Fichero | Algoritmo | Acciones | Mapa |
|---------|-----------|----------|------|
| `dqn_duckie_agent.zip` | DQN | Discretas (5) | `loop_empty-v0` |
| `ppo_duckie_agent.zip` | PPO | Continuas | `loop_empty-v0` |
| `best_duckie_agent.zip` | PPO / SAC (auto) | Continuas | `loop_empty-v0` |

Los vídeos se guardan en `results/Fase_2/videos/`.

`gym-duckietown` es incompatible con Python >= 3.12 (kernel de Colab). Todo el código de
simulación y render corre en un **subproceso Python 3.11** bajo `xvfb-run`, igual que el
entrenamiento. El notebook en sí (kernel 3.12) solo instala dependencias, escribe el script
y muestra los vídeos resultantes.

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO     = 'Aprendizaje-por-Refuerzo-y-Conducci-n-Aut-noma'
    REPO_ABS = f'/content/{REPO}'
    if not os.path.exists(REPO_ABS):
        os.system(f'git clone https://github.com/JaviCeronn/{REPO}.git {REPO_ABS}')
    os.chdir(REPO_ABS)
    os.system('git pull origin main')
    PY = '/usr/bin/python3.11'
else:
    venv_win  = os.path.join(os.getcwd(), '.venv', 'Scripts', 'python.exe')
    venv_unix = os.path.join(os.getcwd(), '.venv', 'bin', 'python')
    if os.path.exists(venv_win):
        PY = venv_win
    elif os.path.exists(venv_unix):
        PY = venv_unix
    else:
        PY = sys.executable
        print('AVISO: .venv no encontrado.')

REPO_ROOT  = os.getcwd()
MODELS     = os.path.join(REPO_ROOT, 'models')
VIDEOS_OUT = os.path.join(REPO_ROOT, 'results', 'Fase_2', 'videos')

print(f'Entorno    : {"Google Colab" if IN_COLAB else "Local"}')
print(f'Python     : {PY}')
print(f'Raiz       : {REPO_ROOT}')
print(f'Modelos    : {MODELS}')
print(f'Videos out : {VIDEOS_OUT}')
if not IN_COLAB:
    print('Nota: Duckietown requiere Linux/WSL2 con Xvfb.')

---
## Instalación de Python 3.11
Igual que en `Fase2_DQN_PPO.ipynb`: `gym-duckietown@daffy` es incompatible con Python 3.12,
así que instalamos 3.11 en paralelo vía deadsnakes y lo usamos exclusivamente en subprocesos.

In [ ]:
if IN_COLAB:
    print('[1/3] software-properties-common...')
    os.system('sudo apt-get install -y -qq software-properties-common > /dev/null')
    print('[2/3] PPA deadsnakes + apt-get update...')
    os.system('sudo add-apt-repository -y ppa:deadsnakes/ppa > /dev/null 2>&1')
    os.system('sudo apt-get update -qq')
    print('[3/3] Python 3.11...')
    os.system('sudo apt-get install -y -qq python3.11 python3.11-venv python3.11-dev python3.11-distutils > /dev/null')
    os.system('wget -q https://bootstrap.pypa.io/get-pip.py -O /tmp/get-pip.py')
    os.system('python3.11 /tmp/get-pip.py -q')
    ret = os.system('python3.11 --version')
    print('>>> Python 3.11 listo <<<' if ret == 0 else 'ERROR al instalar Python 3.11')
else:
    v = sys.version.split()[0]
    print(f'Python local: {v}')
    print('OK' if v.startswith('3.11') else f'AVISO: se recomienda Python 3.11 (actual {v})')

---
## Dependencias
Se instalan sobre Python 3.11 del sistema, no sobre el kernel 3.12. Las versiones están
fijadas en `requirements.txt` (igual que en el entrenamiento) para garantizar compatibilidad.

In [ ]:
if IN_COLAB:
    print('[1/3] Paquetes sistema (OpenGL, Xvfb)...')
    os.system('sudo apt-get install -y -qq xvfb freeglut3-dev libosmesa6-dev '
              'libgl1-mesa-dri libgl1-mesa-glx libglu1-mesa libturbojpeg > /dev/null')
    print('[2/3] requirements.txt en Python 3.11...')
    os.system(f'{PY} -m pip install -q -r requirements.txt')
    print('[3/3] gym-duckietown@daffy (--no-deps)...')
    os.system(f'{PY} -m pip install -q --no-deps '
              'git+https://github.com/duckietown/gym-duckietown.git@daffy')
    ret = os.system(f'{PY} -c "import gym_duckietown, numpy; print(\'OK numpy\', numpy.__version__)"')
    print('>>> Dependencias listas <<<' if ret == 0 else 'ERROR — revisa el output')
else:
    print('Local: ejecuta uv sync y luego:')
    print('  pip install --no-deps git+https://github.com/duckietown/gym-duckietown.git@daffy')

---
## Script de generación de vídeos
Se escribe `gen_videos.py` en `src/` exactamente igual que `Fase2_DQN_PPO.ipynb` escribe
`train.py`: el kernel 3.12 solo genera el fichero; Python 3.11 lo ejecuta en el subproceso.

Las clases `DuckieWrapper`, `DiscreteWrapper` y `CustomCNN` son **idénticas** a las de
`train.py` — condición necesaria para que `model.load()` funcione sin errores de clase.

In [ ]:
import pathlib

_GEN_VIDEOS_SRC = '''\
# gen_videos.py -- genera videos de evaluacion (DQN, PPO, best_agent)
# Lanzar con: xvfb-run -a python3.11 gen_videos.py --models models/ --videos-out results/Fase_2/videos/
#
# IMPORTANTE: los entornos Duckietown-*-v0 (daffy) usan DuckietownEnv, cuya
# accion es [velocidad, angulo_giro]. DISCRETE_ACTIONS y SpeedSteerWrapper
# deben ser IDENTICOS a los de train.py para que los modelos actuen igual.
from __future__ import annotations
import argparse
import io as _io
import logging
import os
import pathlib
import sys
import warnings

os.environ['PYTHONWARNINGS'] = 'ignore'
warnings.filterwarnings('ignore')
logging.disable(logging.INFO)

import cv2
import numpy as np
import torch
import torch.nn as nn
import gym as old_gym
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import DQN, PPO, SAC
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack

IMG_SIZE  = 64
N_STACK   = 4
OBS_SHAPE = (1, IMG_SIZE, IMG_SIZE)
SEED      = 42

# Accion = [velocidad, angulo de giro] (semantica de DuckietownEnv).
DISCRETE_ACTIONS = np.array([
    [0.44,  0.0],
    [0.30,  2.0],
    [0.30, -2.0],
    [0.20,  5.0],
    [0.20, -5.0],
], dtype=np.float32)


# ---------------------------------------------------------------
# Clases IDENTICAS a train.py (obligatorio para model.load())
# ---------------------------------------------------------------

class DuckieWrapper(gym.Env):
    metadata = {"render_modes": ["rgb_array"]}

    def __init__(self, env_name="Duckietown-loop_empty-v0", seed=None):
        super().__init__()
        _out, sys.stdout = sys.stdout, _io.StringIO()
        try:
            import gym_duckietown  # noqa: F401
        finally:
            sys.stdout = _out
        logging.disable(logging.INFO)
        self.env = old_gym.make(env_name)
        if seed is not None:
            try:
                self.env.seed(seed)
            except Exception:
                pass
        self.action_space = spaces.Box(
            low=np.array([-1.0, -1.0], dtype=np.float32),
            high=np.array([1.0, 1.0], dtype=np.float32),
            dtype=np.float32,
        )
        self.observation_space = spaces.Box(
            low=0, high=255, shape=OBS_SHAPE, dtype=np.uint8
        )

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        obs = self.env.reset()
        if isinstance(obs, tuple):
            obs = obs[0]
        return self._process_obs(obs), {}

    def step(self, action):
        action = np.asarray(action, dtype=np.float32).reshape(-1)
        obs, reward, done, info = self.env.step(action)
        return self._process_obs(obs), float(reward), bool(done), False, info

    def _process_obs(self, obs):
        obs  = obs[obs.shape[0] // 2:, :, :]
        gray = cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)
        return np.expand_dims(
            cv2.resize(gray, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA), 0
        ).astype(np.uint8)

    def render(self):
        return self.env.render(mode="rgb_array")

    def close(self):
        self.env.close()


class DiscreteWrapper(gym.ActionWrapper):
    def __init__(self, env):
        super().__init__(env)
        self.action_space = spaces.Discrete(len(DISCRETE_ACTIONS))

    def action(self, action):
        return DISCRETE_ACTIONS[int(action)]


class SpeedSteerWrapper(gym.ActionWrapper):
    """a = [v, s] en [-1,1]^2 -> [velocidad, angulo]. Identico a train.py."""

    def __init__(self, env):
        super().__init__(env)
        self.action_space = spaces.Box(
            low=np.array([-1.0, -1.0], dtype=np.float32),
            high=np.array([1.0, 1.0], dtype=np.float32), dtype=np.float32)

    def action(self, a):
        a = np.asarray(a, dtype=np.float32).reshape(-1)
        vel = 0.25 + 0.15 * float(np.clip(a[0], -1.0, 1.0))
        angle = 5.0 * float(np.clip(a[1], -1.0, 1.0))
        return np.array([vel, angle], dtype=np.float32)


class CustomCNN(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim=256):
        super().__init__(observation_space, features_dim)
        n = observation_space.shape[0]
        self.cnn = nn.Sequential(
            nn.Conv2d(n, 32, 8, 4), nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2), nn.ReLU(),
            nn.Conv2d(64, 64, 3, 1), nn.ReLU(),
            nn.Flatten(),
        )
        with torch.no_grad():
            n_flat = self.cnn(
                torch.as_tensor(observation_space.sample()[None]).float()
            ).shape[1]
        self.linear = nn.Sequential(nn.Linear(n_flat, features_dim), nn.ReLU())

    def forward(self, x):
        return self.linear(self.cnn(x.float() / 255.0))


# ---------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------

def _make_eval_env(map_name, discrete=False):
    def _init():
        e = DuckieWrapper(map_name, seed=SEED)
        return DiscreteWrapper(e) if discrete else SpeedSteerWrapper(e)
    vec = DummyVecEnv([_init])
    return VecFrameStack(vec, n_stack=N_STACK)


def _get_duckie(vec_env):
    # envs[0] es DiscreteWrapper o SpeedSteerWrapper; .env es DuckieWrapper
    return vec_env.envs[0].env


def _load_model(zip_path, algo_classes):
    path = str(zip_path.with_suffix(''))
    for cls in algo_classes:
        try:
            m = cls.load(path)
            print('[load] ' + zip_path.name + ' -> ' + cls.__name__, flush=True)
            return m, cls
        except Exception:
            pass
    raise RuntimeError('No se pudo cargar ' + zip_path.name
                       + ' como ninguno de ' + str([c.__name__ for c in algo_classes]))


def _record_episode(model, map_name, output_path, max_steps, fps, discrete):
    import imageio
    env    = _make_eval_env(map_name, discrete=discrete)
    duckie = _get_duckie(env)
    obs    = env.reset()
    frames, total_reward = [], 0.0

    for step in range(max_steps):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, _ = env.step(action)
        total_reward += float(reward[0])
        frame = duckie.render()
        if frame is not None and isinstance(frame, np.ndarray):
            frames.append(frame)
        if done[0]:
            print('[video] Terminado en paso ' + str(step + 1), flush=True)
            break

    env.close()
    print('[video] Recompensa: ' + str(round(total_reward, 2))
          + '  frames: ' + str(len(frames)), flush=True)

    if not frames:
        print('[video] Sin frames — omitido.', flush=True)
        return False

    out = pathlib.Path(output_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    imageio.mimsave(str(out), frames, fps=fps)
    size_mb = round(out.stat().st_size / 1024 ** 2, 1)
    dur_s   = round(len(frames) / fps, 0)
    print('[video] Guardado: ' + out.name
          + '  (' + str(size_mb) + ' MB, ' + str(dur_s) + 's @ '
          + str(fps) + ' fps)', flush=True)
    return True


# ---------------------------------------------------------------
# Main
# ---------------------------------------------------------------

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--models',     required=True,  help='Directorio con los .zip de modelos')
    parser.add_argument('--videos-out', required=True,  help='Directorio de salida para los MP4')
    parser.add_argument('--steps',      type=int, default=800)
    parser.add_argument('--fps',        type=int, default=30)
    parser.add_argument('--map',        default='Duckietown-loop_empty-v0')
    args = parser.parse_args()

    try:
        from pyvirtualdisplay import Display
        Display(visible=False, size=(1024, 768)).start()
        print('[gen_videos] Display virtual (Xvfb) iniciado.', flush=True)
    except Exception as exc:
        print('[gen_videos] Sin display virtual: ' + str(exc), flush=True)

    models_dir = pathlib.Path(args.models)
    videos_dir = pathlib.Path(args.videos_out)
    videos_dir.mkdir(parents=True, exist_ok=True)

    # Cada entrada prueba varios nombres de fichero (agente final o _v2)
    PLAN = [
        (('dqn_duckie_agent', 'dqn_duckie_v2'),   (DQN,),     True,  args.map, 'dqn_video.mp4'),
        (('ppo_duckie_agent', 'ppo_duckie_v2'),   (PPO,),     False, args.map, 'ppo_video.mp4'),
        (('best_duckie_agent', 'best_agent_v2'),  (PPO, SAC), False, args.map, 'best_video.mp4'),
    ]

    generated, skipped = [], []
    for stems, algo_classes, discrete, map_name, vid_name in PLAN:
        zip_file = None
        for stem in stems:
            cand = models_dir / (stem + '.zip')
            if cand.exists():
                zip_file = cand
                break
        print('', flush=True)
        print('=' * 55, flush=True)
        if zip_file is None:
            print('[gen_videos] OMITIDO: ' + str(stems) + ' no en ' + str(models_dir),
                  flush=True)
            skipped.append(stems[0])
            continue
        print('[gen_videos] ' + zip_file.stem, flush=True)
        try:
            model, _ = _load_model(zip_file, algo_classes)
            ok = _record_episode(
                model, map_name, str(videos_dir / vid_name),
                args.steps, args.fps, discrete,
            )
        except Exception as exc:
            print('[gen_videos] ERROR: ' + str(exc), flush=True)
            ok = False
        (generated if ok else skipped).append(zip_file.stem)

    print('', flush=True)
    print('=' * 55, flush=True)
    print('[gen_videos] Generados: ' + str(generated), flush=True)
    print('[gen_videos] Omitidos:  ' + str(skipped), flush=True)
    for p in sorted(videos_dir.glob('*.mp4')):
        print('  ' + p.name + '  ('
              + str(round(p.stat().st_size / 1024 ** 2, 1)) + ' MB)', flush=True)


if __name__ == '__main__':
    main()
'''

_script = pathlib.Path(REPO_ROOT) / 'src' / 'gen_videos.py'
_script.parent.mkdir(parents=True, exist_ok=True)
_script.write_text(_GEN_VIDEOS_SRC, encoding='utf-8')
print(f'gen_videos.py escrito: {_script}')
print(f'Lineas: {len(_GEN_VIDEOS_SRC.splitlines())}')

---
## Modelos disponibles
Verifica que los `.zip` generados por `Fase2_DQN_PPO.ipynb` existan en `results/Fase_2/`.
Los que falten serán omitidos en la generación de vídeos.

In [ ]:
import pathlib

md = pathlib.Path(MODELS)
md.mkdir(parents=True, exist_ok=True)

TARGETS = [
    ('dqn_duckie_agent.zip',  'DQN  — discreto, 5 acciones'),
    ('ppo_duckie_agent.zip',  'PPO  — continuo'),
    ('best_duckie_agent.zip', 'Mejor agente (PPO o SAC)'),
]

print(f'=== Modelos en {MODELS} ===')
found = 0
for fname, desc in TARGETS:
    p = md / fname
    if p.exists():
        size_mb = round(p.stat().st_size / 1024**2, 1)
        print(f'  [OK     ]  {fname}  ({size_mb} MB) — {desc}')
        found += 1
    else:
        print(f'  [FALTA  ]  {fname} — {desc}')

print()
if found == 0:
    print('Sin modelos. Ejecuta Fase2_DQN_PPO.ipynb primero.')
elif found < 3:
    print(f'{found}/3 modelos encontrados. Los que falten seran omitidos.')
else:
    print('Todos los modelos disponibles — listo para generar videos.')

---
## Generación de vídeos
Lanza `gen_videos.py` bajo `xvfb-run -a python3.11`. El subproceso:
1. Inicia un display virtual Xvfb para el renderizado OpenGL de Duckietown.
2. Carga cada `.zip` con `DQN.load()` / `PPO.load()` / `SAC.load()` (auto-detecta para `best`).
3. Ejecuta hasta 800 pasos @ 30 fps (~26 s de vídeo) en `loop_empty-v0`.
4. Guarda los MP4 en `results/Fase_2/videos/`.

Cada vídeo tarda ~2-5 min en Colab (CPU). La salida se muestra línea a línea en tiempo real.

In [ ]:
import subprocess

STEPS  = 800
FPS    = 30
MAP    = 'Duckietown-loop_empty-v0'
GEN_PY = os.path.join(REPO_ROOT, 'src', 'gen_videos.py')

def _run_streaming(cmd):
    proc = subprocess.Popen(
        cmd, shell=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    return proc.returncode

print('=== Generacion de videos ===')
print(f'  Script     : {GEN_PY}')
print(f'  Modelos    : {MODELS}')
print(f'  Videos out : {VIDEOS_OUT}')
print(f'  Pasos/video: {STEPS}  |  FPS: {FPS}  |  Mapa: {MAP}')
print()

if IN_COLAB:
    os.system('pkill -9 -f Xvfb 2>/dev/null; sleep 1')
    ret = _run_streaming(
        f'xvfb-run -a -s "-screen 0 1024x768x24" '
        f'{PY} -u "{GEN_PY}" '
        f'--models "{MODELS}" --videos-out "{VIDEOS_OUT}" '
        f'--steps {STEPS} --fps {FPS} --map "{MAP}"'
    )
    print(f'\nCodigo de salida: {ret}')
    if ret == 0:
        print('>>> OK — videos en results/Fase_2/videos/ <<<')
    else:
        print('FALLO — revisa el output de arriba.')
else:
    print('Local (Linux/WSL2) — ejecuta en terminal:')
    print(f'  xvfb-run -a {PY} "{GEN_PY}" --models "{MODELS}" --videos-out "{VIDEOS_OUT}" --steps {STEPS} --fps {FPS} --map "{MAP}"')
    print('Windows: Duckietown no soporta Windows sin WSL2.')

---
## Vídeos generados

In [ ]:
import pathlib
from IPython.display import Video, display

LABELS = {
    'dqn_video.mp4' : 'DQN  (discreto, 5 acciones) — loop_empty',
    'ppo_video.mp4' : 'PPO  (continuo)             — loop_empty',
    'best_video.mp4': 'Mejor agente (best_duckie_agent) — loop_empty',
}

videos_dir = pathlib.Path(VIDEOS_OUT)
mp4s = sorted(videos_dir.glob('*.mp4')) if videos_dir.exists() else []

if mp4s:
    print(f'=== {len(mp4s)} video(s) generados ===')
    for p in mp4s:
        label   = LABELS.get(p.name, p.stem)
        size_mb = round(p.stat().st_size / 1024**2, 1)
        print(f'\n{label}  ({size_mb} MB)')
        display(Video(str(p), embed=True, width=640))
else:
    print('Sin videos todavia. Ejecuta la celda anterior primero.')
    print(f'Buscando en: {videos_dir}')